In [0]:
%run "/Workspace/Users/vtilakumara@academic.mit.edu.au/config"

In [0]:
# if the job parameter is not set, default to dev
try:
    ENV = dbutils.widgets.get("ENV")
except:
    ENV = "dev"

print(f"Starting pipeline for {ENV}")
print(f"Environment is {ENV}")
print(f"Starting Task 3 - Gold layer creation for price and demand")

In [0]:
from pyspark.sql import functions as F

## Create GOLD Table - Hourly Demand Trends

In [0]:
silver_df = spark.table(SILVER_TABLE)

hourly_demand = (
    silver_df
    .groupBy("region","year","month","hour")
    .agg(
        F.avg("total_demand_mw").alias("avg_demand_mw"),
        F.max("total_demand_mw").alias("max_demand_mw"),
        F.avg("price_rrp").alias("avg_price_rrp"),
        F.max("price_rrp").alias("max_price_rrp")
    )
)

print(hourly_demand)

In [0]:
hourly_demand.write.format("delta").mode("overwrite").saveAsTable(GOLD_DEMAND_TABLE)

In [0]:
display(spark.sql(f"SELECT * FROM {GOLD_DEMAND_TABLE}"))

## Create GOLD TABLE - Price Spike Detection

In [0]:
silver_df = spark.table(SILVER_TABLE)

price_threshold = silver_df.approxQuantile("price_rrp", [0.95], 0.01)[0]

price_spikes  = silver_df.filter(
                    F.col("price_rrp")>price_threshold
                            )

In [0]:
price_spikes.write.format("delta").mode("overwrite").saveAsTable(GOLD_SPIKE_TABLE)
display(spark.sql(f"SELECT * FROM {GOLD_SPIKE_TABLE}"))

## Create GOLD TABLE - Grid Stress Periods

In [0]:
# Load the silver table into a Spark DataFrame
silver_df = spark.table(SILVER_TABLE)

# Calculate the 90th percentile threshold for total_demand_mw
threshold = silver_df.approxQuantile(
    "total_demand_mw",
    [0.9],
    0.01
)[0]

# Filter rows where total_demand_mw is greater than or equal to the threshold
grid_stress = silver_df.filter(
    F.col("total_demand_mw") >= threshold
)

In [0]:
grid_stress.write.format("delta").mode("overwrite").saveAsTable(GOLD_STRESS_TABLE)
display(spark.sql(f"SELECT * FROM {GOLD_STRESS_TABLE}"))

## Create GOLD TABLE - Battery Dispatch Signals

In [0]:
silver_df = spark.table(SILVER_TABLE)

low_price = silver_df.approxQuantile("price_rrp", [0.30], 0.01)[0]
high_price = silver_df.approxQuantile("price_rrp", [0.70], 0.01)[0]

dispatch_df = (silver_df
                .withColumn(
                    "battery_signal",
                    F.when(F.col("price_rrp") <= low_price, "CHARGE")
                    .when(F.col("price_rrp") >= high_price, "DISCHARGE")
                    .otherwise("IDLE")
                )
                    .withColumn(
            "price_level",
            F.when(F.col("price_rrp") <= low_price, "LOW")
            .when(F.col("price_rrp") >= high_price, "HIGH")
            .otherwise("MEDIUM")
        ))

In [0]:
dispatch_df = dispatch_df.select(
    "region",
    "settlement_timestamp",
    "price_rrp",
    "total_demand_mw",
    "battery_signal",
    "price_level"
)

In [0]:
dispatch_df.write.format("delta").mode("overwrite").saveAsTable(GOLD_BATTERY_DISPATCH_TABLE)
display(spark.sql(f"SELECT * FROM {GOLD_BATTERY_DISPATCH_TABLE}"))